In [20]:
import openai, json

client = openai.OpenAI()
messages = []

In [21]:
def get_weather(city):
  return '33 degrees celcius.'

FUNCTION_MAP = {
  'get_weather': get_weather,
}

In [22]:
TOOLS = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "A function to get the weather of a city.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The name of the city to get the weather of."
          }
        },
        "required": ["city"]
      }
    }
  }
]

In [23]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
  if message.tool_calls:
    messages.append({
      "role": "assistant",
      "content": message.content or "",
      "tool_calls": [
        {
          "id": tool_call.id,
          "type": "function",
          "function": {
            "name": tool_call.function.name,
            "arguments": tool_call.function.arguments,
          }
        } for tool_call in message.tool_calls
      ]
    })

    for tool_call in message.tool_calls:
      function_name = tool_call.function.name
      arguments = tool_call.function.arguments

      print(f"Calling function: {function_name} with arguments: {arguments}")

      try:
        arguments = json.loads(arguments)
      except json.JSONDecodeError:
        arguments = {}
      
      function_to_run = FUNCTION_MAP.get(function_name)

      result = function_to_run(**arguments)

      print(f"Ran {function_name} with arguments: {arguments} for a result of {result}")
      # ** 은 딕셔너리를 키워드 인자로 바꿔준다. arguments는 딕셔너리인데 함수가 받는건 도시 이름 하나이다.
      messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "name": function_name,
        "content": result,
      })
    
    call_ai()
  else:
    messages.append({"role": "assistant", "content": message.content})
    print(f"AI: {message.content}")
      

def call_ai():
  response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=TOOLS,
  )
  process_ai_response(response.choices[0].message)
  

In [24]:
while True:
  message = input("Send a message to the LLM...")
  if message == "quit" or message == "q":
    break
  else:
    messages.append({"role": "user", "content": message,})
    print(f"User: {message}")
    call_ai()

User: 내 이름은 김민기야
AI: 안녕하세요, 김민기님! 어떻게 도와드릴까요?
User: 내 이름이 뭐야?
AI: 김민기님이십니다! 다른 질문이 있으신가요?
User: 스페인 날씨는 어때?
Calling function: get_weather with arguments: {"city":"Spain"}
Ran get_weather with arguments: {'city': 'Spain'} for a result of 33 degrees celcius.
AI: 스페인의 현재 날씨는 섭씨 33도입니다. 더 궁금한 점이 있으신가요?
User: ㅂ
AI: 더 필요한 정보나 질문이 있으시면 말씀해 주세요!


In [25]:
messages

[{'role': 'user', 'content': '내 이름은 김민기야'},
 {'role': 'assistant', 'content': '안녕하세요, 김민기님! 어떻게 도와드릴까요?'},
 {'role': 'user', 'content': '내 이름이 뭐야?'},
 {'role': 'assistant', 'content': '김민기님이십니다! 다른 질문이 있으신가요?'},
 {'role': 'user', 'content': '스페인 날씨는 어때?'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': 'call_ZbgzCpjN8zfekppmZ4KrcixT',
    'type': 'function',
    'function': {'name': 'get_weather', 'arguments': '{"city":"Spain"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_ZbgzCpjN8zfekppmZ4KrcixT',
  'name': 'get_weather',
  'content': '33 degrees celcius.'},
 {'role': 'assistant', 'content': '스페인의 현재 날씨는 섭씨 33도입니다. 더 궁금한 점이 있으신가요?'},
 {'role': 'user', 'content': 'ㅂ'},
 {'role': 'assistant', 'content': '더 필요한 정보나 질문이 있으시면 말씀해 주세요!'}]